In [5]:
import pandas as pd
from pathlib import Path

In [6]:
parent_dir_path = Path.cwd().parent

In [7]:
supplier_feed_sample = pd.read_csv(parent_dir_path / 'data' / 'supplier_feed.csv', nrows=10000)
product_metadata_sample = pd.read_csv(parent_dir_path / 'data' / 'product_metadata.csv', nrows=10000)

In [8]:
supplier_feed_sample.head()

,part_id,stock_level,cost_price,entry_date
0,SP-236,Low Stock,287.32,2024-06-29
1,SP-325,248,388.87,"Mar 17, 2025"
2,SP-179,82,109,2025-04-14
3,SP-332,171,419.6,2025-05-10
4,SP-268,72,279.69,2024-04-08


In [9]:
product_metadata_sample.head()

,part_id,part_name,category
0,SP-101,Electronic Pump,Electronics
1,SP-102,Engine Sensor,Engine
2,SP-103,Engine Pad Set,Engine
3,SP-104,Filter Rotor,Filters
4,SP-105,Brake Rotor,Brakes


In [10]:
supplier_feed_sample[supplier_feed_sample.duplicated(subset=['part_id'])]

,part_id,stock_level,cost_price,entry_date
25,SP-245,low stock,$74.69,2025-04-16
31,SP-224,357,186.78,"Mar 02, 2024"
36,SP-236,144,$352.37,"Aug 06, 2024"
40,SP-328,415,$406.01,"Jul 20, 2024"
47,SP-116,98,$82.96,2024-07-21T00:00:00
...,...,...,...,...
9995,SP-431,437,314.42,"Feb 18, 2024"
9996,SP-104,Out of Stock,411.78,"May 13, 2025"
9997,SP-241,376,$243.72,01/05/25
9998,SP-173,233,$188.92,04/16/24


In [11]:
# pick one part_id and see what the duplicates look like
supplier_feed_sample.loc[supplier_feed_sample['part_id'] == 'SP-245']

,part_id,stock_level,cost_price,entry_date
15,SP-245,UNAVAILABLE,326.83,2024-08-25
25,SP-245,low stock,$74.69,2025-04-16
113,SP-245,265,262.12,07/13/25
132,SP-245,Low Stock,30,2024-04-21
284,SP-245,NaN,27.61,2024-08-07
1052,SP-245,130,$137.09,"Nov 29, 2024"
3313,SP-245,279,297.94,"Aug 12, 2024"
3437,SP-245,low stock,164,"Apr 22, 2024"
3558,SP-245,496,374.1,08/21/24
4226,SP-245,384,NaN,2024-02-13T00:00:00


In [12]:
supplier_feed_sample['entry_date'] = pd.to_datetime(supplier_feed_sample['entry_date'], errors='coerce', format='mixed')

In [13]:
supplier_feed_sample.head()

,part_id,stock_level,cost_price,entry_date
0,SP-236,Low Stock,287.32,2024-06-29
1,SP-325,248,388.87,2025-03-17
2,SP-179,82,109,2025-04-14
3,SP-332,171,419.6,2025-05-10
4,SP-268,72,279.69,2024-04-08


In [14]:
# check for parsing errors
supplier_feed_sample['entry_date'].isna().sum()

np.int64(0)

In [ ]:
# next we check the price column
test_price_col = supplier_feed_sample['cost_price'].astype('float')

In [16]:
# The above code fails with "could not convert string to float: '$203.60'". This means that we have strings mixed in with our numerical data.
# To clean this we use a simple regular expression to remove the dollar sign
test_price_col = supplier_feed_sample['cost_price'].astype(str).str.replace(r'[$]', '', regex=True).astype('float')

In [17]:
test_price_col.head()

0    287.32
1    388.87
2    109.00
3    419.60
4    279.69
Name: cost_price, dtype: float64

In [18]:
# check for nan values
print(test_price_col.isna().sum())
test_price_col[test_price_col.isna()].head()

490


6     NaN
28    NaN
104   NaN
115   NaN
123   NaN
Name: cost_price, dtype: float64

In [19]:
# check these rows in the dataset to see why the price is NaN
supplier_feed_sample[test_price_col.isna()].head()

,part_id,stock_level,cost_price,entry_date
6,SP-481,308,NaN,2024-07-22
28,SP-362,265,NaN,2024-02-22
104,SP-458,38,NaN,2024-11-11
115,SP-417,347,NaN,2024-07-25
123,SP-186,336,NaN,2024-08-17


### The solution for supplier_feed
Now that I have found how to properly standardize the datatypes across the columns, I will now need to decide how to clean the data.

Given the fact that one of the metrics that needs to be tracked is the number of updates to quantities per month, all entries must be kept, simply handling the invalid elements of entries as they come up.

For the error handling, only two columns can contain errors: stock_level and cost_price. The stock_level column can contain the labels "UNAVAILABLE" and "LOW STOCK", among others. For our purposes, I think that assuming that any non-numerical entries should be treated as zero. The reason for this being that "UNAVAILABLE" clearly indicates zero, and "LOW STOCK" could indicate anything from 100s to dozens to actually depleted stock, depending on the supplier's arbitrary internal conventions that are unknown to us. Thus, we default to zero. For cost_price, a missing value can simply indicate that the stock of an item has changed, but not the price. Consequently, when a price is missing, we can fill the cost_price field with the most recent valid value for the same item, and if no such value exists, then we leave the NaN value to inform us that we need to contact the supplier to obtain pricing information if we have no other entries in our database with that part_id that have a valid price.

In summary, the cleaning and formatting pipeline will function as such:
- transform the cost_price and entry_date columns to the correct formats and datatypes
- default all invalid stock_level entries to 0
- sort by part_id

In [20]:
# checks for product_metadata
product_metadata_sample[product_metadata_sample.duplicated(subset=['part_id'])]

,part_id,part_name,category


In [21]:
print(f'{product_metadata_sample['part_name'].isna().sum()} errors in part_name')
print(f'{product_metadata_sample['category'].isna().sum()} errors in category')

0 errors in part_name
0 errors in category


### Testing the pipeline to be used in the script

In [50]:
supplier_feed = pd.read_csv(parent_dir_path / 'data' / 'supplier_feed.csv')
supplier_feed['entry_date'] = pd.to_datetime(supplier_feed['entry_date'], errors='coerce', format='mixed')
supplier_feed = supplier_feed.sort_values(by=['part_id', 'entry_date'], ascending=[True, False]).reset_index(drop=True)
supplier_feed['cost_price'] = supplier_feed['cost_price'].astype(str).str.replace(r'[$]', '', regex=True).astype('float')
supplier_feed['stock_level'] = pd.to_numeric(supplier_feed['stock_level'], errors='coerce').fillna(0).astype(int)

In [51]:
supplier_feed[supplier_feed['cost_price'].isna()].head()

,part_id,stock_level,cost_price,entry_date
18,SP-101,188,NaN,2025-04-23
50,SP-101,68,NaN,2024-10-05
72,SP-101,0,NaN,2024-05-07
89,SP-101,240,NaN,2024-01-28
101,SP-102,256,NaN,2025-06-12


In [52]:
supplier_feed[supplier_feed['part_id'] == 'SP-176']

,part_id,stock_level,cost_price,entry_date
5630,SP-176,243,383.85,2025-07-31
5631,SP-176,345,NaN,2025-07-28
5632,SP-176,32,297.11,2025-07-21
5633,SP-176,108,294.37,2025-07-17
5634,SP-176,391,46.84,2025-07-16
...,...,...,...,...
5697,SP-176,418,431.58,2024-03-27
5698,SP-176,403,441.45,2024-03-26
5699,SP-176,0,307.95,2024-03-23
5700,SP-176,0,76.11,2024-02-25


In [53]:
min_dates = supplier_feed.loc[supplier_feed.groupby('part_id')['entry_date'].idxmin()]
min_dates[min_dates['cost_price'].isna()]

,part_id,stock_level,cost_price,entry_date
5701,SP-176,163,NaN,2024-01-20
7712,SP-203,0,NaN,2024-01-05
8243,SP-210,0,NaN,2024-01-27
8464,SP-213,297,NaN,2024-01-01
9902,SP-232,43,NaN,2024-01-08
10759,SP-243,55,NaN,2024-01-16
12190,SP-262,0,NaN,2024-01-11
13602,SP-281,487,NaN,2024-01-01
16329,SP-317,0,NaN,2024-01-02
18328,SP-344,106,NaN,2024-01-05


In [54]:
supplier_feed['cost_price'] = supplier_feed.groupby('part_id')['cost_price'].bfill()

In [55]:
supplier_feed.head()

,part_id,stock_level,cost_price,entry_date
0,SP-101,214,58.69,2025-07-30
1,SP-101,288,299.39,2025-07-23
2,SP-101,82,274.78,2025-07-21
3,SP-101,217,67.54,2025-07-13
4,SP-101,307,64.87,2025-07-10


In [56]:
supplier_feed[supplier_feed['part_id'] == 'SP-176']

,part_id,stock_level,cost_price,entry_date
5630,SP-176,243,383.85,2025-07-31
5631,SP-176,345,297.11,2025-07-28
5632,SP-176,32,297.11,2025-07-21
5633,SP-176,108,294.37,2025-07-17
5634,SP-176,391,46.84,2025-07-16
...,...,...,...,...
5697,SP-176,418,431.58,2024-03-27
5698,SP-176,403,441.45,2024-03-26
5699,SP-176,0,307.95,2024-03-23
5700,SP-176,0,76.11,2024-02-25
